# tnWF demo — wavefunction flows via tensor networks

A minimalistic, self-contained tour of the V-step methods released in this repo,
reproducing two figures from the paper *"Exponentially-accelerated simulations of
wavefunction flows via tensor networks."*

- **Part 1** — `Dense` vs `TCI+1TDVP` on a 3-D Gaussian mixture, compared by a
  t-SNE overlay against the target (reproduces **Fig. 5**, left / *d*=3 panel).
- **Part 2** — a guided `TCI+2TDVP` run at *d*=8, σ=0.5, *K*=40, showing the
  accuracy / bond-dimension / cost behavior (reproduces the **2TDVP** curve of
  **Fig. 7**).

Everything here is driven by the **closed-form (analytic) Gaussian-mixture
potential** (`V_source="analytic"`), so no neural-network training is required and
the results are deterministic.

> **Scope note.** Paper Fig. 7 also features **MPS-V** (the velocity potential
> pre-trained directly as a tensor train, fed to the V-step with *no runtime
> cross*) and **1TDVP+SE** (one-site TDVP with demand-driven subspace expansion).
> Those methods are **not part of this minimal release**; Part 2 therefore uses
> the shipped `tci_tdvp2` as the two-site-TDVP path. See the closing note.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from tnwf.pipelines.run_evolution import run


# Joint t-SNE over a list of point clouds; returns one embedding per cloud.
# Fitting on the concatenation puts every cloud in a shared 2-D frame so the
# overlays are directly comparable (mirrors scripts/make_fig3_tsne_grid.py).
def tsne_embed(clouds, seed=0):
    joined = np.concatenate(clouds, axis=0)
    perplexity = min(30, max(5, joined.shape[0] // 5))
    emb = TSNE(
        n_components=2, init="pca", perplexity=perplexity,
        random_state=seed, learning_rate="auto",
    ).fit_transform(joined)
    out, i = [], 0
    for c in clouds:
        out.append(emb[i:i + c.shape[0]])
        i += c.shape[0]
    return out

## Part 1 — Dense vs TCI+1TDVP on the 3-D Gaussian mixture

We evolve the same Gaussian source into a 3-D orthogonal Gaussian-mixture target
two ways:

- **`dense`** — the exact $O(N^d)$ reference (feasible at $d=3$, $N=16$);
- **`tci_tdvp1`** — the tensor-network V-step (TT-cross MPO + one-site TDVP).

Each `run(...)` returns per-step sliced-Wasserstein (`sw`), the bond dimension
(`chi_max`), and the final sample cloud (`samples_T`) alongside a `target` draw.

In [ ]:
common = dict(dataset="gmm_3d", V_source="analytic", N=16, K=16,
              n_samples=1500, save=False)

res_dense = run(method="dense", **common)
res_tdvp1 = run(method="tci_tdvp1", method_kwargs={"D_V": 16}, **common)

print(f"Dense      final SW = {res_dense['sw'][-1]:.3f}  (exact reference)")
print(f"TCI+1TDVP  final SW = {res_tdvp1['sw'][-1]:.3f}  "
      f"(chi* = {int(res_tdvp1['chi_max'].max())})")

A **joint t-SNE overlay** of the final samples shows that both methods reproduce
the target's cluster geometry — this is the *d*=3 panel of Fig. 5.

In [ ]:
target = res_dense["target"]
emb_t, emb_d, emb_v = tsne_embed(
    [target, res_dense["samples_T"], res_tdvp1["samples_T"]], seed=0)

fig, ax = plt.subplots(figsize=(6, 6), dpi=110)
ax.scatter(emb_t[:, 0], emb_t[:, 1], s=22, alpha=0.35, c="0.6", label="Target")
ax.scatter(emb_d[:, 0], emb_d[:, 1], s=16, alpha=0.70, c="#1f77b4",
           label=f"Dense  (SW={res_dense['sw'][-1]:.2f})")
ax.scatter(emb_v[:, 0], emb_v[:, 1], s=16, alpha=0.70, c="#d62728",
           label=f"TCI+1TDVP  (SW={res_tdvp1['sw'][-1]:.2f})")
ax.set(title="3-D Gaussian mixture — final samples (t-SNE overlay)\n"
             "reproduces paper Fig. 5 (left, d=3)", xticks=[], yticks=[])
ax.legend(loc="upper right", framealpha=0.9)
plt.tight_layout()
plt.show()

## Part 2 — Tutorial: TCI+2TDVP at *d*=8, σ=0.5, *K*=40

This is the high-dimensional showcase. At $d=8$ the dense grid has $N^8$ amplitudes
— far out of reach — so only the tensor-network V-step is viable. We use the
**two-site** integrator `tci_tdvp2`, which lets the MPS bond dimension grow via SVD
and then truncates it back, so $\chi$ adapts to the state.

The knobs:

| argument | meaning | value here |
|---|---|---|
| `dataset="gmm_8d"` | 8-D orthogonal GMM; σ=0.5 is its built-in width | d=8, σ=0.5 |
| `K` | number of Trotter steps (time resolution) | **40** |
| `N` | grid points per axis | 8 *(see note)* |
| `method_kwargs["D_max"]` | MPS bond-dimension cap | 16 |
| `method_kwargs["D_V"]` | TT-cross rank for the $e^{i\beta V}$ MPO | 16 |

> **Runtime / resolution note.** The paper runs a higher grid resolution; we set
> `N=8` here so the full *K*=40 sweep finishes in **~2–3 min on a laptop CPU**.
> The qualitative Fig. 7 behavior — the self-limiting bond dimension and the
> cost/accuracy trend — is preserved. Raise `N` for a sharper (slower) run.

In [ ]:
res2 = run(method="tci_tdvp2", dataset="gmm_8d", V_source="analytic",
           N=8, K=40, n_samples=1500,
           method_kwargs={"D_max": 16, "D_V": 16}, save=False)

print(f"d=8, sigma=0.5, K=40")
print(f"  final SW                    = {res2['sw'][-1]:.3f}")
print(f"  chi* (self-limiting bond)   = {int(res2['chi_max'].max())}")
print(f"  total wall-clock            = {res2['step_times'].sum():.0f} s")

The three panels below mirror the reproducible parts of Fig. 7: **(a)** accuracy
per step, **(c)** the bond dimension self-limiting near $\chi^\star\approx 16$
(pure two-site TDVP truncates the subspace each step), and **(d)** the accuracy
reached versus accumulated wall-clock cost.

In [ ]:
steps = np.arange(len(res2["sw"]))
cum_time = np.concatenate([[0.0], np.cumsum(res2["step_times"])])

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), dpi=110)

axes[0].plot(steps, res2["sw"], "-o", ms=3)
axes[0].set(xlabel="Trotter step $k$", ylabel=r"SW($\psi_k$)", title="(a) accuracy")

axes[1].plot(steps, res2["chi_max"], "-s", ms=3, c="C2")
axes[1].axhline(16, ls="--", c="0.5", lw=1)
axes[1].set(xlabel="Trotter step $k$", ylabel=r"$\chi_{\max}$",
            title=r"(c) bond dimension (self-limits $\approx$16)")

axes[2].plot(cum_time, res2["sw"], "-o", ms=3, c="C3")
axes[2].set(xlabel="wall-clock [s]", ylabel=r"SW($\psi_k$)",
            title="(d) cost vs accuracy")

fig.suptitle(r"TCI+2TDVP on the $d{=}8$ GMM ($\sigma{=}0.5$, $K{=}40$) "
             "— cf. paper Fig. 7", y=1.04)
plt.tight_layout()
plt.show()

A t-SNE overlay of the *d*=8 final samples against the target confirms the
tensor-network preparation recovers the mixture structure in eight dimensions:

In [ ]:
emb_t8, emb_v8 = tsne_embed([res2["target"], res2["samples_T"]], seed=0)

fig, ax = plt.subplots(figsize=(5.5, 5.5), dpi=110)
ax.scatter(emb_t8[:, 0], emb_t8[:, 1], s=20, alpha=0.35, c="0.6", label="Target")
ax.scatter(emb_v8[:, 0], emb_v8[:, 1], s=16, alpha=0.70, c="#d62728",
           label=f"TCI+2TDVP  (SW={res2['sw'][-1]:.2f})")
ax.set(title="d=8 GMM — TCI+2TDVP samples (t-SNE overlay)", xticks=[], yticks=[])
ax.legend(loc="upper right", framealpha=0.9)
plt.tight_layout()
plt.show()

### What this notebook does *not* reproduce

Two elements of paper Fig. 7 rely on code outside this minimal release:

- the **SW − SW$_\mathrm{cl}$** deviation panels (b) require the classical-flow
  reference trajectory;
- the **1TDVP+SE** comparison curve and the **MPS-V** trained-tensor-train V-step
  (the direct-core *bypass* with no runtime cross).

Those methods are part of the research codebase but not this release. Here we
demonstrate the shipped `dense`, `tci_tdvp1`, and `tci_tdvp2` V-steps end to end.
See the [README](../README.md) and the paper for the full set of methods.